# Weekly Assessment - 18 Nov

In [1]:
#Q1
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X, y = load_wine(return_X_y=True, as_frame=True)

# (a)
print("Shape of X:", X.shape)
print("Class counts:\n", y.value_counts().sort_index())

# (b)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

# (c) fit on train only, transform on test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# (d)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
pred = lr.predict(X_test_s)
print("Test accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Shape of X: (178, 13)
Class counts:
 target
0    59
1    71
2    48
Name: count, dtype: int64
Train shape: (124, 13)  Test shape: (54, 13)
Test accuracy: 0.9814814814814815
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54



**Q1 answer:** Precision for class 0 means: out of all the wines the model labelled as class 0, this is the fraction that really were class 0.

In [2]:
#Q2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# (a)
dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("Decision Tree accuracy:", dt.score(X_test, y_test))

# (b)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print("Random Forest accuracy:", rf.score(X_test, y_test))

# (c)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 3 features:")
for name, score in imp.head(3).items():
    print(f"  {name}: {score:.4f}")

Decision Tree accuracy: 0.9629629629629629
Random Forest accuracy: 1.0
Top 3 features:
  color_intensity: 0.1802
  flavanoids: 0.1659
  alcohol: 0.1420


**Q2 answer:** A forest averages the votes of many different trees, so the random mistakes of any single tree cancel out and it overfits less.

In [3]:
#Q3
from sklearn.neighbors import KNeighborsClassifier

# (a) unscaled
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
acc_raw = knn_raw.score(X_test, y_test)
print("KNN unscaled accuracy:", acc_raw)

# (b) scaled
knn_sc = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
acc_sc = knn_sc.score(X_test_s, y_test)
print("KNN scaled accuracy:", acc_sc)

# (c)
print(f"Unscaled: {acc_raw:.4f} | Scaled: {acc_sc:.4f} | Difference: {acc_sc-acc_raw:+.4f}")

KNN unscaled accuracy: 0.7407407407407407
KNN scaled accuracy: 0.9629629629629629
Unscaled: 0.7407 | Scaled: 0.9630 | Difference: +0.2222


**Q3 answer:** KNN works on raw distances, so a big-valued feature like proline (hundreds) drowns out small ones; a decision tree only asks 'is this feature above a threshold?' one feature at a time, so the units do not matter.

In [4]:
#Q4
from sklearn.cluster import KMeans

# (a) no y used anywhere
X_all_s = StandardScaler().fit_transform(X)

# (b)
km = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = km.fit_predict(X_all_s)

# (c)
print("Rows per cluster:")
print(pd.Series(labels).value_counts().sort_index())

# (d)
df = X.copy()
df["cluster"] = labels
print(df.groupby("cluster")[["alcohol", "proline", "color_intensity"]].mean())

Rows per cluster:
0    65
1    51
2    62
Name: count, dtype: int64
           alcohol      proline  color_intensity
cluster                                         
0        12.250923   510.169231         2.973077
1        13.134118   619.058824         7.234706
2        13.676774  1100.225806         5.453548


**Q4 answer:** Based on the averages: **Cluster 0** = 'Light and mild wines' (lowest alcohol, lowest proline, pale colour). **Cluster 1** = 'Deep-coloured medium wines' (medium alcohol and proline but by far the highest colour intensity). **Cluster 2** = 'Strong, rich wines' (highest alcohol and a very high proline).

In [5]:
#Q5
import csv, json

# create the file
with open("students.csv", "w", newline="") as f:
    f.write("name,marks,city\nAsha,78,Bangalore\nRavi,,Chennai\nMeena,91,Bangalore\nImran,64,Hyderabad\nDivya,88,\n")

def read_marks(path):
    # (a)
    try:
        f = open(path, newline="")
    except FileNotFoundError:
        print(f"File not found: {path}")
        return None
    # (b)
    with f:
        rows = list(csv.DictReader(f))
    # (c)
    with_marks = [float(r["marks"]) for r in rows if r["marks"].strip() != ""]
    summary = {
        "total_rows": len(rows),
        "rows_with_marks": len(with_marks),
        "average_marks": sum(with_marks) / len(with_marks) if with_marks else 0,
    }
    # (d)
    with open("summary.json", "w") as out:
        json.dump(summary, out, indent=2)
    with open("summary.json") as back:
        print(back.read())
    return summary

print("Call 1: students.csv")
print(read_marks("students.csv"))
print("\nCall 2: missing.csv")
print(read_marks("missing.csv"))

Call 1: students.csv
{
  "total_rows": 5,
  "rows_with_marks": 4,
  "average_marks": 80.25
}
{'total_rows': 5, 'rows_with_marks': 4, 'average_marks': 80.25}

Call 2: missing.csv
File not found: missing.csv
None
